# 2026 TDL Challenge: GraphUniverse

This notebook evaluates your model across a grid of GraphUniverse's synthetic graph distributions:
- **Homophily** (low, mid, high)
- **Average degree** (low, high)  
- **Power-law exponent** (low, high)

Each configuration is trained with multiple random seeds. The best checkpoint from each run is then evaluated on all other grid settings (out-of-distribution evaluation).

## Setup Requirements

**Make sure Weights & Biases is configured:**
```bash
wandb login
```

## How to Use

1. Set your `MODEL_CONFIG` (path to your model yaml)
2. Run the evaluation
3. Results will be saved in:
   - `results.json` with detailed metrics
   - Heatmap visualizations showing performance across the grid
   - OOD performance delta plots

In [1]:
import sys
from pathlib import Path

# Setup paths
_ROOT = Path.cwd().resolve()
_REPO = _ROOT if (_ROOT / "configs" / "run.yaml").exists() else _ROOT.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from utils import (
    resolve_project_root,
    run_challenge_grid,
    check_challenge_grid,
    save_challenge_artifacts,
)

PROJECT_ROOT = resolve_project_root(_REPO)

## Configuration

**Set your model config path here:**

In [2]:
# Your model configuration (e.g., "graph/gcn", "graph/gin", "graph/gat")
MODEL_CONFIG = "graph/gin"

## Sanity check

This check ensures that no notebook cells have been modified below this point. Most submissions should only change the model MODEL_CONFIG variable above.

In [ ]:
expected_hash = "f87b2cfd0c61cabaa50ee2c05d8fa7b5e1d78e14e8c3a50dfdd933a5b389bfde"

In [7]:
# UNIQUE_HASH_MARKER
import json
import hashlib

def hash_remaining_cells(notebook_path: str, marker_string: str = "# UNIQUE_HASH_MARKER") -> str:
    """
    Computes a SHA-256 hash of all cells from the marker cell to the end of the notebook.

    Args:
        notebook_path (str): The hardcoded path to the notebook file.
        marker_string (str, optional): A unique string present in the source of the calling cell.
                                       Defaults to "# UNIQUE_HASH_MARKER".

    Returns:
        str: The SHA-256 hash of the content of the target cells.

    Raises:
        ValueError: If the marker_string is not found in the notebook data.
    """
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_data = json.load(f)

    cells = notebook_data.get('cells', [])
    start_index = -1

    for i, cell in enumerate(cells):
        source = "".join(cell.get('source', []))
        if marker_string in source[:len(marker_string)]:
            start_index = i
            break

    if start_index == -1:
        raise ValueError(f"Marker string '{marker_string}' not found in the notebook.")

    content_to_hash = ""
    for cell in cells[start_index:]:
        content_to_hash += "".join(cell.get('source', []))

    return hashlib.sha256(content_to_hash.encode('utf-8')).hexdigest()

hash_value = hash_remaining_cells(notebook_path="run_evaluation.ipynb")

print(f"Computed hash: {hash_value}")

if hash_value != expected_hash:
    raise ValueError("❌ The content of the notebook has been modified. Please ensure that you are using the original notebook without any changes from this point onward.")
else:
    print("✅ Notebook content is verified.")


Computed hash: f87b2cfd0c61cabaa50ee2c05d8fa7b5e1d78e14e8c3a50dfdd933a5b389bfde


ValueError: ❌ The content of the notebook has been modified. Please ensure that you are using the original notebook without any changes from this point onward.

## Check runs

Here we quickly check that all configurations run correctly before running the full evaluation. The first time this will also create all the needed datasets.

In [ ]:
check_challenge_grid(
    project_root=PROJECT_ROOT,s
    model_config=MODEL_CONFIG,
    quiet=False,
)

Seed set to 42
Seed set to 42


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃    ┃ Name                                  ┃ Type                  ┃ Params ┃ Mode  ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0  │ feature_encoder                       │ AllCellFeatureEncoder │  1.1 K │ train │
│ 1  │ feature_encoder.encoder_0             │ BaseEncoder           │  1.1 K │ train │
│ 2  │ feature_encoder.encoder_0.BN          │ GraphNorm             │     45 │ train │
│ 3  │ feature_encoder.encoder_0.linear      │ Linear                │  1.0 K │ train │
│ 4  │ feature_encoder.encoder_0.relu        │ ReLU                  │      0 │ train │
│ 5  │ feature_encoder.encoder_0.dropout     │ Dropout               │      0 │ train │
│ 6  │ backbone                              │ GNNWrapper            │  8.4 K │ train │
│ 7  │ backbone.backbone                     │ GIN                   │  8.3 K │ train │
│ 8  │ backbone.backbone.dropout             │ Dropout               │      0 │ train │
│ 9  │ backbone.backbone.act                 │ ReLU                  │      0 │ train │
│ 10 │ backbone.backbone.convs               │ ModuleList            │  8.3 K │ train │
│ 11 │ backbone.backbone.convs.0             │ GINConv               │  8.3 K │ train │
│ 12 │ backbone.backbone.convs.0.aggr_module │ SumAggregation        │      0 │ train │
│ 13 │ backbone.backbone.convs.0.nn          │ MLP                   │  8.3 K │ train │
│ 14 │ backbone.backbone.convs.0.nn.lins     │ ModuleList            │  8.3 K │ train │
│ 15 │ backbone.backbone.convs.0.nn.lins.0   │ Linear                │  4.2 K │ train │
│ 16 │ backbone.backbone.convs.0.nn.lins.1   │ Linear                │  4.2 K │ train │
│ 17 │ backbone.backbone.convs.0.nn.norms    │ ModuleList            │      0 │ train │
│ 18 │ backbone.backbone.convs.0.nn.norms.0  │ Identity              │      0 │ train │
│ 19 │ backbone.backbone.norms               │ ModuleList            │      0 │ train │
│ 20 │ backbone.backbone.norms.0             │ Identity              │      0 │ train │
│ 21 │ backbone.backbone._trim               │ TrimToLayer           │      0 │ train │
│ 22 │ backbone.ln_0                         │ LayerNorm             │    128 │ train │
│ 23 │ readout                               │ NoReadOut             │  1.3 K │ train │
│ 24 │ readout.linear                        │ Linear                │  1.3 K │ train │
│ 25 │ val_acc_best                          │ MeanMetric            │      0 │ train │
└────┴───────────────────────────────────────┴───────────────────────┴────────┴───────┘

Trainable params: 10.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 26                                                                                          
Modules in eval mode: 0

Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val/accuracy        │    0.15464289486408234    │
│         val/auroc         │    0.7083438038825989     │
│         val/loss          │    2.7606165409088135     │
│       val/precision       │    0.15481530129909515    │
│        val/recall         │    0.15794607996940613    │
└───────────────────────────┴───────────────────────────┘

...

Trainable params: 9.6 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 9.6 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 26                                                                                          
Modules in eval mode: 0

Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val/loss          │       68177.6015625       │
│          val/mae          │    187.28530883789062     │
│          val/mse          │       63497.0859375       │
└───────────────────────────┴───────────────────────────┘

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/loss         │      115559.2109375       │
│         test/mae          │    231.76828002929688     │
│         test/mse          │        112235.5625        │
└───────────────────────────┴───────────────────────────┘

## Run Evaluation

This will train and evaluate your model across all grid configurations.

In [ ]:
results, study_id = run_challenge_grid(
    project_root=PROJECT_ROOT,
    model_config=MODEL_CONFIG,
)

## Save Results

Generate JSON output and visualization plots.

In [ ]:
output_paths = save_challenge_artifacts(
    results,
    model_config=MODEL_CONFIG,
    study_id=study_id,
)

print(f"\nResults saved to: {output_paths['dir']}")
print(f"JSON: {output_paths['json']}")

## Inspect Results

View results as a DataFrame for quick inspection.

In [ ]:
import pandas as pd

pd.DataFrame(results)